In [70]:
import importlib
import utils
importlib.reload(utils)

import numpy as np

In [71]:
n = 5
steps = 12               # number of steps required for time evolution

def u(x):
    return np.sin(2*np.pi*2*x) + 0.5*np.sin(2*np.pi*7*x)

nu = 1e-3                 # diffusion coefficient 
save_every = 200          # after this many steps, take a snapshot of the function to plot on the graph
cfl = 0.1                 # controls time step relative to grid spacing. affects stability of time0-step scheme

In [72]:
N     = 2**n
x     = np.linspace(0, 1, N, endpoint=False)

dx    = x[1] - x[0] 
dt    = cfl * dx*dx / nu 
u0    = u(x)

The result below show that when converted to matrices, the MPOs are almost exactly the same.

In [73]:
L = utils.laplacian(N, dx, "dirichlet", "dense")

A_dense = utils.time_step(L, 1, dt, nu)
A_svd = utils.mat_to_qtt_mpo(A_dense, n)
A_manual = utils.qtt_diffusion_mpo(n, cfl)

A_svd_dense = np.asarray(A_svd.to_dense()).reshape(N, N)
A_manual_dense = np.asarray(A_manual.to_dense()).reshape(N, N)

print("||A_svd - A_dense|| =", np.linalg.norm(A_svd_dense - A_dense))
print("||A_manual - A_dense|| =", np.linalg.norm(A_manual_dense - A_dense))
print("||A_manual - A_svd||  =", np.linalg.norm(A_manual_dense - A_svd_dense))

||A_svd - A_dense|| = 4.27863164248793e-15
||A_manual - A_dense|| = 1.6014269617894046e-15
||A_manual - A_svd||  = 4.4258996490073184e-15


The two code cells below show that the manual MPO explodes in bond dimension very quickly, which leads to much longer computation times.

In [74]:
mps0 = utils.vec_to_qtt_mps(u0, n)
saved, bonds = utils.evolve_mps_timed(mps0, A_manual, steps)

step  0: 0.001394 s, max bond = 12
step  1: 0.002835 s, max bond = 36
step  2: 0.082108 s, max bond = 64
step  3: 0.152674 s, max bond = 64
step  4: 0.474765 s, max bond = 64
step  5: 0.952036 s, max bond = 64
step  6: 0.773145 s, max bond = 64
step  7: 1.547084 s, max bond = 64
step  8: 3.941252 s, max bond = 64
step  9: 9.812980 s, max bond = 64
step 10: 27.958368 s, max bond = 64
step 11: 88.515569 s, max bond = 64


In [75]:
mps0 = utils.vec_to_qtt_mps(u0, n)
saved, bonds = utils.evolve_mps_timed(mps0, A_svd, steps)

step  0: 0.002786 s, max bond = 4
step  1: 0.001357 s, max bond = 4
step  2: 0.000726 s, max bond = 4
step  3: 0.000797 s, max bond = 4
step  4: 0.000707 s, max bond = 4
step  5: 0.001360 s, max bond = 4
step  6: 0.000744 s, max bond = 4
step  7: 0.000705 s, max bond = 4
step  8: 0.000685 s, max bond = 4
step  9: 0.000870 s, max bond = 4
step 10: 0.000869 s, max bond = 4
step 11: 0.001058 s, max bond = 4


Below I take a look at the tensor shape and realise that they are quite different.

In [76]:
print("A_manual bond sizes:", A_manual.bond_sizes())
print("A_svd bond sizes:", A_svd.bond_sizes())


for i, T in enumerate(A_manual):
    print(f"A_manual tensor {i}: shape = {T.shape}")

for i, T in enumerate(A_svd):
    print(f"A_svd tensor {i}: shape = {T.shape}")

A_manual bond sizes: [5, 5, 5, 5, 3]
A_svd bond sizes: [3, 3, 3, 3]
A_manual tensor 0: shape = (3, 5, 2, 2)
A_manual tensor 1: shape = (5, 5, 2, 2)
A_manual tensor 2: shape = (5, 5, 2, 2)
A_manual tensor 3: shape = (5, 5, 2, 2)
A_manual tensor 4: shape = (5, 3, 2, 2)
A_svd tensor 0: shape = (2, 2, 3)
A_svd tensor 1: shape = (3, 2, 2, 3)
A_svd tensor 2: shape = (3, 2, 2, 3)
A_svd tensor 3: shape = (3, 2, 2, 3)
A_svd tensor 4: shape = (3, 2, 2)


Even after compression, the manual MPO remains the same.

In [77]:
A_manual.compress(cutoff=1e-12)
print("A_manual bond sizes:", A_manual.bond_sizes())
for i, T in enumerate(A_manual):
    print(f"A_manual tensor {i}: shape = {T.shape}")

A_manual bond sizes: [5, 5, 5, 5, 3]
A_manual tensor 0: shape = (3, 5, 2, 2)
A_manual tensor 1: shape = (5, 5, 2, 2)
A_manual tensor 2: shape = (5, 5, 2, 2)
A_manual tensor 3: shape = (5, 5, 2, 2)
A_manual tensor 4: shape = (5, 3, 2, 2)


Conclusion: `from.dense` optimises the MPO in such a way that my manual construction is not able to emulate